# 3장. Live Demo (Finger Classification)
훈련된 CNN 모델(`best_model_finger.pth`)을 로드하여 카메라 영상에서 사람의 손가락 편 개수를 실시간으로 감지하고 시연합니다.

In [1]:
import torchvision
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import PIL.Image
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import traitlets
from jetbot import Camera, bgr8_to_jpeg

# 모델 아키텍처 및 가중치 불러오기
model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 6)
model.load_state_dict(torch.load('best_model_finger.pth'))

device = torch.device('cuda')
model = model.to(device)
model = model.eval().half() # 연산속도를 높이기 위해 반정밀도(fp16) 모델로 설정
print('Model loaded successfully!')

Model loaded successfully!


## 전처리 함수 정의
카메라로부터의 BGR 입력을 모델 입력 규격에 맞게 변환하고 정규화합니다.

In [2]:
mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess(image):
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

## 실시간 감지 UI 생성 및 배치

In [3]:
camera = Camera()
image_widget = widgets.Image(format='jpeg', width=224, height=224)
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

# UI Widgets
pred_label = widgets.HTML(value="<h2 style='color: #2196F3;'>Predicted: -</h2>")
prob_bars = [widgets.FloatProgress(value=0.0, min=0.0, max=1.0, description=f'Class {i}:') for i in range(6)]

ui_box = widgets.VBox([
    pred_label,
    *prob_bars
])

display(widgets.HBox([image_widget, ui_box]))

## 실시간 추론 함수 연결
카메라 화면이 갱신될 때마다 자동으로 예측하여 UI에 보여줍니다.

In [4]:
def execute(change):
    try:
        image = change['new']
        output = model(preprocess(image)).detach().float()
        probabilities = F.softmax(output, dim=1).cpu().numpy().flatten()
        pred_class = np.argmax(probabilities)
        
        # UI 업데이트
        pred_label.value = f"<h2 style='color: #4CAF50;'>Predicted: {pred_class} fingers</h2>"
        for i in range(6):
            prob_bars[i].value = float(probabilities[i])
    except Exception as e:
        import traceback
        pred_label.value = f"<pre style='color: red;'>Error: {traceback.format_exc()}</pre>"

# 테스트: 메인 스레드에서 먼저 한 번 수동 실행하여 CUDA 컨텍스트 워밍업 및 에러 확인
execute({'new': camera.value})

camera.observe(execute, names='value')
print('Real-time inference started!')

Real-time inference started!


## 프로젝트 종료하기
시연이 완료되면 실시간 감지를 끄고 카메라 자원을 반환합니다.

In [5]:
import time
camera.unobserve(execute, names='value')
time.sleep(0.1)
camera.stop()
print('Camera demo stopped successfully!')

: 

: 

: 